# DS 227 &middot; Knowledge Discovery in Data &mdash; Week 3 Lab
## Parsing HTML with BeautifulSoup

Most of the world's data lives inside web pages, wrapped in HTML that was written for
a browser to draw &mdash; not for you to analyse. This lab is the first half of scraping:
**turning a page's tags into rows you can work with.**

**How long:** about 45 minutes. **What you need:** a browser. Nothing to install &mdash;
`BeautifulSoup` already ships with Colab.

We parse a small HTML string held in a variable. Next week you fetch real pages over the
network; this week we keep the page fixed so the focus stays on **parsing**.

---
## Part 0 &middot; The page

Run the cell. This is a tiny, **invented** directory page &mdash; the kind of markup a
real site would bury inside a hundred other tags. Read the HTML, not just the output.

In [ ]:
html = """
<div class="directory">
  <h1>Barangay Health Stations &mdash; Cebu City</h1>
  <ul>
    <li class="station" data-open="1988">
      <a href="/station/lahug">Lahug Health Center</a>
      <span class="staff">6 staff</span>
    </li>
    <li class="station" data-open="1995">
      <a href="/station/mabolo">Mabolo Health Center</a>
      <span class="staff">5 staff</span>
    </li>
    <li class="station" data-open="2003">
      <a href="/station/talamban">Talamban Health Center</a>
      <span class="staff">4 staff</span>
    </li>
    <li class="station featured" data-open="1979">
      <a href="/station/guadalupe">Guadalupe Health Center</a>
      <span class="staff">6 staff</span>
    </li>
  </ul>
</div>
"""

print(html)

---
## Part 1 &middot; From text to a tree

To a browser, that string is a **tree**: a `<ul>` with `<li>` children, each holding an
`<a>` and a `<span>`. BeautifulSoup parses the text into that tree so you can walk it.

Run the cell.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

# .find returns the FIRST match; .get_text() pulls the text out of a tag
first = soup.find("li", class_="station")
print("First station block:\n", first)
print()
print("Just its link text:", first.find("a").get_text())

**Answer here** (double-click to edit):

1. `soup.find("li")` returned only **one** `<li>`, though the page has four. Which one,
   and what rule did `.find` follow to choose it?
   &rarr; *your answer*

2. `.get_text()` gave you `Lahug Health Center` without the `<a>` tags around it. In one
   sentence, what does `.get_text()` do?
   &rarr; *your answer*

---
## Part 2 &middot; Every match, not just the first

`.find_all` returns a **list** of every matching tag. Looping over it is how you turn a
page into rows. Run the cell.

In [ ]:
stations = soup.find_all("li", class_="station")
print("Found", len(stations), "stations\n")

for s in stations:
    name = s.find("a").get_text()
    staff = s.find("span", class_="staff").get_text()
    print(f"{name:32}  {staff}")

**Answer here:**

1. `.find` and `.find_all` sound alike. State the difference in the **type** each one
   returns, and why that decides whether you can loop over the result.
   &rarr; *your answer*

2. The last station, Guadalupe, has `class="station featured"` &mdash; two classes. It
   still matched `class_="station"`. Why?
   &rarr; *your answer*

---
## Part 3 &middot; Attributes are data too

Text is not the only thing worth extracting. The `href` on each link and the
`data-open` on each `<li>` are **attributes** &mdash; you read them like a dictionary.
Run the cell.

In [ ]:
rows = []
for s in stations:
    link = s.find("a")
    rows.append({
        "name": link.get_text(),
        "url": link["href"],          # attribute access, like a dict
        "opened": s["data-open"],
        "staff": s.find("span", class_="staff").get_text(),
    })

import pandas as pd
table = pd.DataFrame(rows)
table

**Answer here:**

1. `link["href"]` reads an attribute; `link.get_text()` reads the text between the tags.
   For `<a href="/station/lahug">Lahug Health Center</a>`, name what each one returns.
   &rarr; *your answer*

2. The `opened` and `staff` columns are strings like `"1988"` and `"6 staff"`, not
   numbers. Which KDD phase (from Week 2) is this &mdash; and whose job is it to fix?
   &rarr; *your answer*

---
## Part 4 &middot; Why scraping is fragile

Your code in Part 3 assumed every station has an `<a>`, a `data-open`, and a
`<span class="staff">`. Real pages break that assumption constantly.

Run the cell &mdash; it adds one malformed station &mdash; and watch it fail.

In [ ]:
broken = html.replace(
    '<li class="station" data-open="1995">\n      <a href="/station/mabolo">Mabolo Health Center</a>',
    '<li class="station">\n      <a>Mabolo Health Center</a>',
)
bsoup = BeautifulSoup(broken, "html.parser")

for s in bsoup.find_all("li", class_="station"):
    print(s.find("a")["href"])   # one station now has no href

**Answer here:**

1. Copy the **last line** of the error. Which station caused it, and what was missing?
   &rarr; *your answer*

2. `link.get("href")` returns `None` for a missing attribute instead of crashing, where
   `link["href"]` raises. When would you *want* the crash rather than the quiet `None`?
   &rarr; *your answer*

3. One missing tag stopped the whole loop. What does that tell you about trusting a
   scraper you ran once and never checked again?
   &rarr; *your answer*

---
## Stretch &mdash; optional

Stop here if you like; the required part is done.

### Stretch 1 &middot; Make it survive missing data

Rewrite the Part 3 loop so a station with no `<a href>` still produces a row &mdash; with
`None` (or `""`) for the url instead of crashing. Use `.get()` and check for `None`.

In [ ]:
# your code here

### Stretch 2 &middot; Select with a CSS selector

`soup.select("li.station.featured a")` uses the same selector syntax as CSS. Use it to
pull just the **featured** station's link text, and say in one line what the selector means.

In [ ]:
# your code here

&rarr; *what does the selector mean?*

---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to
download.

You need a **submit token** &mdash; one covers every lab for a month. Open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token), sign in and generate it,
then add it **once** to Colab's Secrets panel (the &#128273; icon, left sidebar) as
`LATARAK_TOKEN`. After that the cell reads it automatically, with no prompt. No Secrets
panel? The cell will just ask, hiding what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "ds227", 3

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/ds227/lab/3/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

# A month-long token. Store it once in Colab Secrets (key LATARAK_TOKEN) and
# this reads it with no prompt; otherwise it asks and hides what you type.
try:
    from google.colab import userdata
    token = (userdata.get("LATARAK_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 3 submission page](https://portal.latarak.com/course/ds227/lab/3/submit) and upload it.

Re-submitting replaces your previous attempt; the most recent version is the one kept.